# 🧠 NeuroScan AI — Google Colab Interactive Notebook
### Brain Tumour & Alzheimer's Detection System with Explainable AI (Grad-CAM)

> ⚠️ **Medical AI Disclaimer**: This system is developed for **research and educational purposes only**. It is not a certified medical device and must not be used as a substitute for professional medical diagnosis or clinical judgment.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Install Dependencies
PyTorch and torchvision are already pre-installed in Google Colab. We just need to install medical imaging, reporting, and web interface packages.

In [ ]:
!pip install -q pydicom nibabel SimpleITK fpdf2 streamlit fastapi uvicorn python-multipart loguru

## 3. Clone / Setup Project Repository
If running from a GitHub clone, pull your repository. Otherwise, ensure the `brain-mri-ai` package is added to Python path.

In [ ]:
import sys, os
# If cloned from github in colab:
# !git clone https://github.com/<your-username>/brain-mri-ai.git
# %cd brain-mri-ai

# Add current directory to Python module search path
sys.path.insert(0, os.path.abspath("."))
print("Python path updated. Current working dir:", os.getcwd())

## 4. Generate Synthetic Datasets
Creates realistic synthetic brain MRI scans with simulated lesions and atrophy patterns to verify the full pipeline end-to-end without needing gigabytes of external downloads.

In [ ]:
!python scripts/generate_synthetic_data.py --output_dir data/synthetic --num_per_class 40

## 5. Train Brain Tumour Classifier on Colab GPU
Runs transfer learning with ResNet-50 / EfficientNet, patient-aware splitting (no data leakage), early stopping, and automatic test evaluation.

In [ ]:
!python scripts/train.py \
    --task tumour \
    --backbone resnet50 \
    --data_dir data/synthetic/tumour \
    --output_dir models/registry/tumour_colab_run \
    --epochs 10 \
    --batch_size 16 \
    --lr 0.0005 \
    --device cuda

## 6. Inspect Evaluation Results & Model Card
View the generated confusion matrix, bootstrap 95% confidence intervals, and the clinical model card.

In [ ]:
from IPython.display import Image, Markdown, display

# Display confusion matrix
cm_path = "models/registry/tumour_colab_run/confusion_matrix.png"
if os.path.exists(cm_path):
    display(Image(cm_path))

# Display Model Card markdown
card_path = "models/registry/tumour_colab_run/MODEL_CARD.md"
if os.path.exists(card_path):
    with open(card_path, "r") as f:
        display(Markdown(f.read()))

## 7. Interactive Prediction & Grad-CAM Heatmap Visualization
Test an individual MRI scan directly inside the notebook and visualize the Grad-CAM heatmap overlay with uncertainty estimation.

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from src.ml.models.classifier import ModelFactory
from src.ml.evaluation.inference import InferencePipeline

# Load best trained model
checkpoint_path = "models/registry/tumour_colab_run/best_model.pt"
model = ModelFactory.create("tumour", "resnet50", pretrained=False)
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
model.eval()

pipeline = InferencePipeline(model=model, task="tumour", mc_dropout_samples=5)

# Pick a test image
sample_path = "data/synthetic/tumour/glioma/patient0000_0000.png"
res = pipeline.predict(sample_path)

print(f"Predicted Class: {res.predicted_class} (Confidence: {res.confidence * 100:.1f}%)")
print(f"Uncertainty Score: {res.uncertainty_score} | Low Confidence Flag: {res.is_low_confidence}")

# Display Side-by-Side: Scan vs Grad-CAM Overlay
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(res.original_image)
axes[0].set_title("Original MRI Scan")
axes[0].axis("off")

if res.overlay is not None:
    axes[1].imshow(res.overlay)
    axes[1].set_title(f"Grad-CAM Overlay ({res.predicted_class})")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

## 8. Launch the Streamlit Web Application from Google Colab
We can launch the interactive Streamlit UI in the background and expose it using `localtunnel` or `cloudflared` to access the full web interface directly in your browser.

In [ ]:
# Install localtunnel to tunnel Streamlit port
!npm install -g localtunnel

# Fetch public IP for password access on localtunnel
import urllib.request
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"Your LocalTunnel password (Endpoint IP) is: {public_ip}")

# Launch Streamlit in the background
!nohup streamlit run src/frontend/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

# Expose port 8501 through localtunnel
!npx localtunnel --port 8501